In [1]:
import pandas as pd
import numpy as np
import time
import chess
import chess.engine
import asyncio
from pathfinding.core.diagonal_movement import DiagonalMovement
from pathfinding.core.grid import Grid
from pathfinding.finder.a_star import AStarFinder
from IPython.display import clear_output, display
pd.set_option('display.max_rows', None)    # Show all rows
pd.set_option('display.max_columns', None) # Show all columns

In [2]:
def erase_piece(board,position):

    x = position[0]
    y = position[1]

    if isinstance(x, str):
        x = board.columns.get_loc(x)
        y = board.index.get_loc(y)
    
    for i in range(-1,2,1):
        for j in range(-1,2,1):
            # hoje aprendi que i==0 & j==0 é diferente de (i==0) & (j==0)

            board.iat[y + i, x + j] = '.'

    return board

def draw_piece(board, position, piece):

    x = position[0] # column
    y = position[1] # line
    
    if isinstance(x, str):
        x = board.columns.get_loc(x)
        y = board.index.get_loc(y)
        
    for i in range(-1,2,1):
        for j in range(-1,2,1):
            # hoje aprendi que i==0 & j==0 é diferente de (i==0) & (j==0)
            if i == 0 and j == 0:
                board.iat[y + i, x + j] = piece
                
            else:
                board.iat[y + i, x + j] = '#'
    return board
    

In [3]:
def tabuleiro_grade37x37():
    # Gerar Nomes das Colunas (Margens + Peças + Gaps = 37 colunas)
    # 3 no início
    colunas = ['E0', 'E1', 'E2'] 
    letras = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
    p_count = 1
    
    for i in range(8):
        # Insere o bloco da peça (ex: a0, a, a1)
        colunas.extend([f"{letras[i]}0", letras[i], f"{letras[i]}1"])
        
        # Insere o gap de 1 coluna (p1, p2, p3...), exceto após a coluna 'h'
        if i < 7:
            colunas.extend([f"p{p_count}"])
            p_count += 1
            
    # no final
    colunas.extend(['D0', 'D1', 'D2'])
    
    # Gera Nomes das Linhas (Idêntico às colunas, usando L1, L2...)
    linhas = ['Top0', 'Top1', 'Top2']
    numeros = ['8', '7', '6', '5', '4', '3', '2', '1']
    l_count = 1
    
    for i in range(8):
        # Bloco da peça (ex: 8_0, 8, 8_1)
        linhas.extend([f"{numeros[i]}_0", numeros[i], f"{numeros[i]}_1"])
        
        # Gap de 1 linhas (L1, L2, L3...)
        if i < 7:
            linhas.extend([f"L{l_count}"])
            l_count += 1
            
    linhas.extend(['Bot0', 'Bot1', 'Bot2'])
    
    # Criar o DataFrame 37x37 preenchido com ""
    matriz_base = np.full((37, 37), '', dtype='<U4')
    tabuleiro = pd.DataFrame(matriz_base, index=linhas, columns=colunas)

    return tabuleiro

def preencher_tabuleiro(grid37x37,fen):
    tabuleiro = grid37x37
    tabuleiro[:] = "."
    

    # le as posições das pecas no tabuleiro 8x8 (FEN)
    posicao = fen.split(' ')[0]
    linhas_fen = posicao.split('/')
    
    # Distribui as peças com o novo espaçamento
    nu_linha = 0
    for l in linhas_fen:
        nu_colum = 0

        # 4 * posição 
        novo_l = (4 * nu_linha) + 4
        for c in l:
            
            if c.isdigit():
                nu_colum += int(c)

            else:
                novo_c = (4 * nu_colum) + 4
                # Preenche ao redor com '#' e o centro com a peca
                tabuleiro = draw_piece(tabuleiro, [novo_c, novo_l], c)
                        
                nu_colum += 1              
        nu_linha += 1
    return tabuleiro


In [4]:
#facil demais usando a biblioteca pathfinding. Talvez eu tente implementar o codigo eu mesmo
# para ter algum desafio.
def calcular_rota_a_star(tabuleiro_df, coordenada_inicio, coordenada_fim):

    matriz_str = tabuleiro_df.to_numpy()
    
    #  Onde for '.' vira 1 (Livre), o resto vira 0
    matriz_binaria = np.where(matriz_str == '.', 1, 0).tolist()
    
    # Inicio e fim precisa estar livre
    if isinstance(coordenada_inicio[0],str):
        y_ini = tabuleiro_df.index.get_loc(coordenada_inicio[1]) 
        x_ini = tabuleiro_df.columns.get_loc(coordenada_inicio[0]) 
        y_fim = tabuleiro_df.index.get_loc(coordenada_fim[1]) 
        x_fim = tabuleiro_df.columns.get_loc(coordenada_fim[0])        
    else:
        y_ini, x_ini = coordenada_inicio
        y_fim, x_fim = coordenada_fim
    
    matriz_binaria[y_ini][x_ini] = 1
    matriz_binaria[y_fim][x_fim] = 1 
    
    # Grid da biblioteca
    grid = Grid(matrix=matriz_binaria)
    
    # (coluna, linha) e nao (linha, coluna) como np.array
    start = grid.node(x_ini, y_ini)
    end = grid.node(x_fim, y_fim)
    # tem 4 opçoes para configurar a diagonal
    #   always = 1
    #    never = 2
    #    if_at_most_one_obstacle = 3
    #    only_when_no_obstacle = 4
    finder = AStarFinder(diagonal_movement=DiagonalMovement.if_at_most_one_obstacle)
    
    caminho, execucoes = finder.find_path(start, end, grid)
    
    # Retorna a lista de coordenadas (X, Y) do caminho
    return caminho


In [5]:
def visualizar_rota(tabuleiro_df, caminho_nodes):
    
    tab_visual = tabuleiro_df.copy()
    
    linhas_reais = tab_visual.index.tolist()
    colunas_reais = tab_visual.columns.tolist()
    
    # Substitui cada passo da rota por um '*'
    for node in caminho_nodes:
        # Extrai X (coluna) e Y (linha)
        x_col = node.x
        y_lin = node.y
        
        # mapeia de volta para os nomes originais (ex: 'a', 'p1', '4_0')
        nome_linha = linhas_reais[y_lin]
        nome_coluna = colunas_reais[x_col]
        
        # Só marca com '*' se não for a própria peça (para não apagar a letra da peça)
        if tab_visual.at[nome_linha, nome_coluna] == '.':
            tab_visual.at[nome_linha, nome_coluna] = '*'
            
    return tab_visual



In [ ]:
 async def main() -> None:
     transport, engine = await chess.engine.popen_uci(r"/home/eros/virtualenvs/Chess/Stockfish/src/stockfish")

     board = chess.Board()
     fen = board.fen()
    
    # cria tableiro expandido e preenche coma possição
     board_expanded = tabuleiro_grade37x37()
     board_expanded = preencher_tabuleiro(board_expanded, fen)   
     
     while not board.is_game_over():
         
         display(board_expanded)
         
         # formato uci
         jogador = input()
         if not board.is_legal(chess.Move.from_uci(jogador)):
             clear_output(wait=True)
             continue
         x0, y0, x1, y1 = jogador
         peca = board_expanded.at[y0,x0]
         print(peca)
         board_expanded = erase_piece(board_expanded, [x0,y0])
         board_expanded = erase_piece(board_expanded, [x1,y1])
         rota = calcular_rota_a_star(board_expanded, [x0,y0], [x1,y1])
         
         clear_output(wait=False)
         
         
         board_expanded = draw_piece(board_expanded, [x1,y1],peca)
         display(board_expanded)
         
         

         # Jogada da pessoa 
         board.push_san(jogador)
         
         clear_output(wait=True)
         
         bot = await engine.play(board, chess.engine.Limit(depth = 20, time=10))
         print(bot.move)
         x0, y0, x1, y1 = str(bot.move)
         peca = board_expanded.at[y0,x0]
         print(peca)
         board_expanded = erase_piece(board_expanded, [x0,y0])
         board_expanded = erase_piece(board_expanded, [x1,y1])
         rota = calcular_rota_a_star(board_expanded, [x0,y0], [x1,y1])
         
         clear_output(wait=False)
         
         
         board_expanded = draw_piece(board_expanded, [x1,y1],peca)
         display(board_expanded)
         # Jogada do bot
         board.push(bot.move)
         clear_output(wait=False)
         
         await asyncio.sleep(1)
         

     await engine.quit()

 await main()

,E0,E1,E2,a0,a,a1,p1,b0,b,b1,p2,c0,c,c1,p3,d0,d,d1,p4,e0,e,e1,p5,f0,f,f1,p6,g0,g,g1,p7,h0,h,h1,D0,D1,D2
Top0,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.
Top1,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.
Top2,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.
8_0,.,.,.,#,#,#,.,#,#,#,.,#,#,#,.,#,#,#,.,#,#,#,.,#,#,#,.,.,.,.,.,#,#,#,.,.,.
8,.,.,.,#,r,#,.,#,n,#,.,#,b,#,.,#,q,#,.,#,k,#,.,#,b,#,.,.,.,.,.,#,r,#,.,.,.
8_1,.,.,.,#,#,#,.,#,#,#,.,#,#,#,.,#,#,#,.,#,#,#,.,#,#,#,.,.,.,.,.,#,#,#,.,.,.
L1,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.
7_0,.,.,.,#,#,#,.,#,#,#,.,#,#,#,.,.,.,.,.,#,#,#,.,#,#,#,.,#,#,#,.,#,#,#,.,.,.
7,.,.,.,#,p,#,.,#,p,#,.,#,p,#,.,.,.,.,.,#,p,#,.,#,p,#,.,#,p,#,.,#,p,#,.,.,.
7_1,.,.,.,#,#,#,.,#,#,#,.,#,#,#,.,.,.,.,.,#,#,#,.,#,#,#,.,#,#,#,.,#,#,#,.,.,.
